# Notebook 02 — Feature Engineering

**Goal:** Transform raw 10-level LOB snapshots into 24+ predictive features.

Features: mid-price, spread, OFI, volume delta, aggressive pressure, VWMP, depth imbalance, volatility regimes.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.utils import set_seed
from src.data_loader import load_fi2010_raw, prepare_labels
from src.features import (
    engineer_all_features, compute_mid_price, compute_spread,
    compute_order_flow_imbalance, compute_volatility_regime
)

set_seed(42)
sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

RESULTS_DIR = Path('../results/plots')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Raw Data

In [ ]:
X, y = load_fi2010_raw('../data/raw')
print(f'X: {X.shape}, y: {y.shape}')

## 2. Run Full Feature Engineering Pipeline

In [ ]:
df_features = engineer_all_features(X, n_levels=10)
print(f'\nEngineered features shape: {df_features.shape}')
print(f'Columns: {list(df_features.columns)}')
df_features.head()

## 3. Feature Correlation Analysis

In [ ]:
key_features = ['mid_price', 'spread', 'ofi', 'volume_delta',
                'aggressive_pressure', 'weighted_mid', 'volatility',
                'depth_imbalance_1', 'log_return']

corr = df_features[key_features].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, ax=ax)
ax.set_title('Engineered Feature Correlations')
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'feature_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Distributions

In [ ]:
plot_features = ['ofi', 'volume_delta', 'aggressive_pressure',
                 'spread', 'volatility', 'log_return']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, feat in zip(axes.flat, plot_features):
    data = df_features[feat].dropna()
    ax.hist(data, bins=80, color='#3498db', edgecolor='white', alpha=0.8)
    ax.set_title(feat)
    ax.axvline(data.mean(), color='red', linestyle='--', alpha=0.7)

fig.suptitle('Feature Distributions', fontsize=14)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. OFI vs Future Mid-Price Direction

In [ ]:
labels = prepare_labels(y, horizon=2)  # k=3 horizon
label_names = {0: 'Down', 1: 'Stationary', 2: 'Up'}

fig, ax = plt.subplots(figsize=(10, 6))
for cls in [0, 1, 2]:
    mask = labels[:len(df_features)] == cls
    ax.hist(df_features['ofi'].values[mask], bins=60, alpha=0.5,
            label=label_names[cls], density=True)
ax.set_xlabel('Order Flow Imbalance')
ax.set_ylabel('Density')
ax.set_title('OFI Distribution by Future Price Direction')
ax.legend()
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'ofi_by_direction.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Volatility Regimes

In [ ]:
vol = df_features['volatility'].values
regimes = df_features['vol_regime'].values
regime_names = {0: 'Low', 1: 'Medium', 2: 'High'}

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Volatility time series colored by regime
colors = {0: '#2ecc71', 1: '#f39c12', 2: '#e74c3c'}
for r in [0, 1, 2]:
    mask = regimes == r
    axes[0].scatter(np.where(mask)[0], vol[mask], s=0.1,
                    c=colors[r], label=regime_names[r])
axes[0].set_title('Volatility with Regime Classification')
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Volatility')
axes[0].legend(markerscale=20)

# Regime distribution
counts = pd.Series(regimes).value_counts().sort_index()
axes[1].bar([regime_names[i] for i in counts.index], counts.values,
            color=[colors[i] for i in counts.index])
axes[1].set_title('Volatility Regime Distribution')
axes[1].set_ylabel('Count')

plt.tight_layout()
fig.savefig(RESULTS_DIR / 'volatility_regimes.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save Processed Features

In [ ]:
# Add labels for the primary horizon (k=3)
labels_k3 = prepare_labels(y, horizon=2)
df_features['label'] = labels_k3[:len(df_features)]

# Also save labels for other horizons
for h_idx, h_name in enumerate(['k1', 'k2', 'k3', 'k5', 'k10']):
    h_labels = prepare_labels(y, horizon=h_idx)
    df_features[f'label_{h_name}'] = h_labels[:len(df_features)]

# Save
save_path = Path('../data/processed/features.parquet')
save_path.parent.mkdir(parents=True, exist_ok=True)
df_features.to_parquet(save_path, index=False)
print(f'Saved: {save_path} ({save_path.stat().st_size / 1e6:.1f} MB)')
print(f'Shape: {df_features.shape}')

# Also save raw X for sequence models
np.save('../data/processed/X_raw.npy', X)
np.save('../data/processed/y_raw.npy', y)
print('Saved: X_raw.npy, y_raw.npy')

## Summary

- Engineered 24+ features from raw 10-level LOB data
- Key features: OFI, volume delta, aggressive pressure, volatility regimes
- Saved processed features to `data/processed/features.parquet`

**Next:** Notebook 03 — Baseline Models